# Experimento 2: ¿sobrevive la escala λ ≈ 0.7 kpc a una validación independiente?

**Proyecto:** *Vacuum Particle Journey*, línea B (consolidar la anomalía).

Siguiendo la revisión, aquí λ deja de llamarse "alcance de los puentes ER" y pasa a ser una **escala candidata de una corrección gravitatoria no local**. La pregunta es solo observacional: ¿existe esa escala, con capacidad predictiva, aunque su origen siga sin conocerse?

**Diseño fijado antes de mirar los resultados (preregistro):**
1. **División fija de la muestra.** Cada galaxia se asigna a *descubrimiento* o *reservada* con un hash de su nombre. La asignación no depende de los datos y es siempre la misma.
2. **Todo se ajusta solo con el grupo de descubrimiento:** λ, g† y los modelos rivales.
3. **El grupo reservado se usa una sola vez**, con λ y g† congelados: la única libertad allí es la masa estelar de cada galaxia, igual en todos los modelos.
4. **Criterio de éxito** (las cuatro a la vez):
   - en el grupo reservado, la versión con λ mejora a λ = 0 con un test de Wilcoxon por galaxias: p < 0.01 con todos los puntos y p < 0.05 sin los interiores (hay menos datos);
   - también mejora a la predicción de MOND completo (λ = 0.27·R_d) y a un modelo combinado;
   - la geometría en el plano del disco encuentra la λ que se espera si la señal esférica es real (calibración cruzada, factor < 2);
   - la mejora no se correlaciona con indicadores instrumentales (inclinación, distancia, muestreo, calidad).
5. Todo se guarda en archivos (`resultados_validacion.json`, `por_galaxia.csv`) con las versiones y una huella de los datos, para que sea reproducible.

**Además:** el límite matemático del modelo con κ constante que señala la revisión, y una sección para aplicar los parámetros congelados a una **muestra externa** (por ejemplo, curvas de alta resolución) cuando la tengas.

In [ ]:
import os, io, re, glob, zipfile, urllib.request, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares, minimize_scalar
from scipy.special import i0, i1, k0, k1
from scipy.stats import spearmanr

warnings.filterwarnings("ignore", category=RuntimeWarning)
RNG = np.random.default_rng(42)
QUICK = False

COL = {"Datos": "#222222", "Solo bariones": "#8a8a85", "NFW ΛCDM (c–M)": "#6250d6",
       "RAR (g† = 1.2e-10)": "#e87ba4", "RAR local (g† global)": "#eb6834", "RAR no local (ER)": "#1baf7a", "Instrumental": "#2a78d6", "Densidad": "#eda100", "Tamaño": "#e34948"}
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.edgecolor": "#c3c2b7", "lines.linewidth": 2, "font.size": 10, "legend.frameon": False,
})

G = 4.30091e-6
C_KMS = 299792.458
H0 = 70.0; H0_KPC = H0 / 1000.0
KPC_M = 3.0857e19
TO_MS2 = 1e6 / KPC_M                    # (km/s)^2/kpc -> m/s^2
try:
    import plotly.graph_objects as go
    HAVE_PLOTLY = True
except ImportError:
    HAVE_PLOTLY = False
    print("plotly no disponible: las figuras 3D se harán con matplotlib (estáticas)")
from scipy import fft as sfft

import hashlib, platform, json, scipy
from scipy.stats import wilcoxon
SEED = 20261119
RNG = np.random.default_rng(SEED)
SPLIT_SALT = "vpj-validacion-2026"
ALPHA_Q = 0.27            # λ de MOND completo ≈ 0.27·R_d (resultado del notebook 6)

## 1. Datos (igual que en los notebooks 4–6)

In [ ]:
DATA_DIR = "sparc_rotmod"
BASES = ["https://astroweb.case.edu/SPARC/", "http://astroweb.cwru.edu/SPARC/"]
ZENODO = "https://zenodo.org/records/16284118/files/Rotmod_LTG.zip?download=1"

def _get(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    return urllib.request.urlopen(req, timeout=60).read()

def _files():
    return sorted(glob.glob(os.path.join(DATA_DIR, "**", "*_rotmod.dat"), recursive=True))

def get_sparc():
    os.makedirs(DATA_DIR, exist_ok=True)
    if _files():
        return True
    for url in [b + "Rotmod_LTG.zip" for b in BASES] + [ZENODO]:
        try:
            print("Descargando", url)
            zipfile.ZipFile(io.BytesIO(_get(url))).extractall(DATA_DIR)
            if _files():
                return True
        except Exception as e:
            print("  falló:", type(e).__name__)
    try:
        from google.colab import files
        print("Sube Rotmod_LTG.zip (de astroweb.case.edu/SPARC)")
        for name, content in files.upload().items():
            zipfile.ZipFile(io.BytesIO(content)).extractall(DATA_DIR)
        return bool(_files())
    except Exception:
        return False

def get_quality():
    for b in BASES:
        try:
            txt = _get(b + "SPARC_Lelli2016c.mrt").decode("latin-1")
            q = {}
            for line in txt.splitlines():
                t = line.split()
                if len(t) >= 18 and re.fullmatch(r"\d+", t[1] or "") and t[17] in ("1", "2", "3"):
                    q[t[0]] = dict(Q=int(t[17]), Inc=float(t[5]))
            if len(q) > 100:
                return q
        except Exception:
            pass
    return None

def load_rotmod(path):
    name = os.path.basename(path).replace("_rotmod.dat", "")
    d = np.atleast_2d(np.loadtxt(path, comments="#"))
    g = dict(name=name, r=d[:, 0], vobs=d[:, 1], err=d[:, 2], vgas=d[:, 3], vdisk=d[:, 4], vbul=d[:, 5])
    ok = (g["r"] > 0) & (g["err"] > 0)
    o = np.argsort(g["r"][ok])
    for k in ["r", "vobs", "err", "vgas", "vdisk", "vbul"]:
        g[k] = g[k][ok][o]
    g["vgas2"] = g["vgas"] * np.abs(g["vgas"])
    g["vstar2"] = g["vdisk"] * np.abs(g["vdisk"]) + 1.4 * g["vbul"] * np.abs(g["vbul"])
    return g

def _v2_expdisk(r, M, Rd):
    y = r / (2 * Rd)
    return 4 * np.pi * G * (M / (2 * np.pi * Rd**2)) * Rd * y**2 * (i0(y) * k0(y) - i1(y) * k1(y))

def _v2_nfw(r, V200, c):
    x = r / (V200 / (10 * H0_KPC))
    mu = lambda y: np.log1p(y) - y / (1 + y)
    return V200**2 * mu(c * x) / (x * mu(c))

def make_synthetic_set(n=30, seed=0):
    rng = np.random.default_rng(seed)
    out = {}
    for name in ["NGC3198", "NGC2403", "DDO154"] + [f"SYN{i:03d}" for i in range(n - 3)]:
        V200 = 10 ** rng.uniform(1.6, 2.4); c = 10 ** rng.normal(1.0, 0.12)
        Rd = rng.uniform(1, 5); Ml = 10 ** rng.uniform(8.5, 11); Mg = Ml * 10 ** rng.uniform(-1, 0.7)
        r = np.linspace(0.3 * Rd, rng.uniform(5, 9) * Rd, rng.integers(12, 40))
        vdisk = np.sqrt(_v2_expdisk(r, Ml, Rd)); vgas = np.sqrt(_v2_expdisk(r, Mg, 2.5 * Rd))
        vtrue = np.sqrt(vgas**2 + 0.5 * vdisk**2 + _v2_nfw(r, V200, c))
        err = 2 + 0.04 * vtrue
        os.makedirs("synthetic", exist_ok=True)
        path = os.path.join("synthetic", f"{name}_rotmod.dat")
        np.savetxt(path, np.c_[r, vtrue + rng.normal(0, err), err, vgas, vdisk, 0 * r], header="Distance = 10 Mpc")
        out[name] = load_rotmod(path)
    return out

HAVE_SPARC = get_sparc()
if HAVE_SPARC:
    GAL = {}
    for f in _files():
        g = load_rotmod(f)
        if len(g["r"]) >= 5:
            GAL[g["name"]] = g
    Q = get_quality()
    if Q:
        GAL = {k: v for k, v in GAL.items() if k in Q and Q[k]["Q"] < 3 and Q[k]["Inc"] >= 30}
        print("Corte de calidad aplicado (Q<3, inc≥30°).")
    else:
        print("Sin tabla de calidad: se usan todas las galaxias.")
else:
    print("⚠️  SIN DATOS REALES: galaxias SINTÉTICAS con halo NFW. Los resultados no tienen valor físico.")
    GAL = make_synthetic_set()

if QUICK and len(GAL) > 40:
    keep = [k for k in ["NGC3198", "NGC2403", "DDO154"] if k in GAL]
    rest = [k for k in GAL if k not in keep]
    GAL = {k: GAL[k] for k in keep + list(RNG.choice(rest, 40 - len(keep), replace=False))}
N_TOT = sum(len(g["r"]) for g in GAL.values())
print(f"Galaxias: {len(GAL)} | puntos: {N_TOT}")

In [ ]:
NG = 400
U_GRID = np.geomspace(0.1, 1.5, 40)
LAM_GRID = np.r_[0.0, np.geomspace(0.1, 100, 24)]                                       # kpc
GD_GRID = np.unique(np.r_[np.geomspace(0.6e-10, 2.4e-10, 25), 1.2e-10]) / TO_MS2        # (km/s)^2/kpc
I_GD12 = int(np.argmin(np.abs(GD_GRID * TO_MS2 - 1.2e-10)))

def vb2(g, u):
    return g["vgas2"] + u * g["vstar2"]

def prior(u):
    return np.log10(u / 0.5) / 0.1

def rar_g(gb, gd):
    return gb / (-np.expm1(-np.sqrt(gb / gd)))

def F_matrix(r, a, lam):
    R, A = r[:, None], a[None, :]
    if lam == 0:
        return (A < R).astype(float)
    x = R / lam
    e_lo = np.exp(-np.clip((A - R) / lam, 0, None))
    e_hi = np.exp(-np.clip((R - A) / lam, 0, None))
    e_sum = np.exp(-(A + R) / lam)
    I_lo = e_lo * (x - 1) + e_sum * (1 + x)
    I_hi = 2 * A / lam - e_hi * (1 + x) + e_sum * (1 + x)
    return np.where(R <= A, I_lo, I_hi) * lam / (2 * A)

def setup(g):
    if "_s" in g:
        return g["_s"]
    rmax = g["r"].max()
    rg = np.linspace(rmax / NG, rmax, NG)
    s = dict(rg=rg, mids=0.5 * (np.r_[0.0, rg[:-1]] + rg),
             vg=np.interp(rg, np.r_[0.0, g["r"]], np.r_[0.0, g["vgas2"]]),
             vs=np.interp(rg, np.r_[0.0, g["r"]], np.r_[0.0, g["vstar2"]]))
    g["_s"] = s
    return s

def phantom_shells(s, U, GD):
    # G·M_fantasma en cascarones -> forma (nΥ, ng†, NG)
    v2 = s["vg"][None, :] + U[:, None] * s["vs"][None, :]
    gb = np.clip(v2, 1e-6, None) / s["rg"][None, :]
    gr = rar_g(gb[:, None, :], GD[None, :, None])
    GM = s["rg"][None, None, :] ** 2 * (gr - gb[:, None, :])
    return np.diff(np.concatenate([np.zeros(GM.shape[:2] + (1,)), GM], axis=2), axis=2)

def model_v2(g, u, lam, gd):
    s = setup(g)
    d = phantom_shells(s, np.array([u]), np.array([gd]))[0, 0]
    return vb2(g, u) + F_matrix(g["r"], s["mids"], lam) @ d / g["r"]

# Comprobaciones: F(λ→0) es un escalón; con λ pequeño se recupera la RAR local exacta
_g = next(iter(GAL.values()))
v_loc = np.sqrt(_g["r"] * rar_g(np.clip(vb2(_g, 0.5), 1e-6, None) / _g["r"], 1.2e-10 / TO_MS2))
v_0 = np.sqrt(model_v2(_g, 0.5, 0.0, 1.2e-10 / TO_MS2))
v_s = np.sqrt(model_v2(_g, 0.5, 1e-3, 1.2e-10 / TO_MS2))
print(f"Test: |v(λ=0) − RAR exacta| máx = {np.max(np.abs(v_0 - v_loc)):.2f} km/s; "
      f"|v(λ=0.001) − v(λ=0)| máx = {np.max(np.abs(v_s - v_0)):.3f} km/s (ambos deben ser ≈ 0)")

In [ ]:
ARCSEC = 206265.0
paths = _files() if HAVE_SPARC else glob.glob(os.path.join("synthetic", "*_rotmod.dat"))
DIST = {}
for f in paths:
    with open(f) as fh:
        for line in fh:
            m = re.search(r"Distance\s*=\s*([\d.]+)", line)
            if m:
                DIST[os.path.basename(f).replace("_rotmod.dat", "")] = float(m.group(1)); break
if not HAVE_SPARC:
    _rd = np.random.default_rng(3)
    DIST = {k: float(10 ** _rd.uniform(0.5, 1.8)) for k in GAL}
    print("⚠️  Distancias sintéticas aleatorias (solo para probar el código)")
GAL = {k: g for k, g in GAL.items() if k in DIST}
NAMES = list(GAL)
D = np.array([DIST[k] for k in NAMES])
RMAX = np.array([GAL[k]["r"].max() for k in NAMES])
MB = np.array([max(GAL[k]["r"][-1] * vb2(GAL[k], 0.5)[-1] / G, 1e5) for k in NAMES])
SIGMA = MB / (np.pi * (RMAX * 1e3) ** 2)          # M☉/pc²
N_TOT_PTS = sum(len(g["r"]) for g in GAL.values())
print(f"{len(NAMES)} galaxias | D: {D.min():.1f}–{D.max():.1f} Mpc | Σ_b: {np.median(SIGMA):.1f} M☉/pc² (mediana)")
rho, p = spearmanr(np.log10(D), np.log10(RMAX))
print(f"Correlación distancia–tamaño en la muestra: ρ = {rho:.2f} (p = {p:.0e})  ← por esto λ instrumental imitaría 'λ ∝ tamaño'")

In [ ]:
def get_rdisk():
    for b in BASES:
        try:
            txt = _get(b + "SPARC_Lelli2016c.mrt").decode("latin-1"); out = {}
            for line in txt.splitlines():
                t = line.split()
                if len(t) >= 18 and re.fullmatch(r"\d+", t[1] or "") and t[17] in ("1", "2", "3"):
                    out[t[0]] = float(t[11])
            if len(out) > 100:
                return out
        except Exception:
            pass
    return {}

RD_TAB = get_rdisk()
RD = []
for k in NAMES:
    g = GAL[k]
    est = g["r"][np.argmax(np.abs(g["vdisk"]))] / 2.2 if np.any(g["vdisk"] != 0) else np.nan
    RD.append(RD_TAB.get(k, est) if RD_TAB.get(k, 0) > 0 else est)
RD = np.array(RD)
RD = np.where(np.isfinite(RD) & (RD > 0), RD, np.nanmedian(RD))
print(f"R_d {'de la tabla de SPARC' if RD_TAB else 'estimado desde V_disk'}: mediana {np.median(RD):.2f} kpc "
      f"(16–84 %: {np.percentile(RD,16):.2f}–{np.percentile(RD,84):.2f})")

In [ ]:
# Huella de reproducibilidad
h = hashlib.sha256()
for f in sorted(paths):
    h.update(open(f, "rb").read())
DATA_HASH = h.hexdigest()[:16]
QT = get_quality() if HAVE_SPARC else None
INC = np.array([QT[k]["Inc"] if QT and k in QT else np.nan for k in NAMES])
QF = np.array([QT[k]["Q"] if QT and k in QT else np.nan for k in NAMES])
VERS = dict(python=platform.python_version(), numpy=np.__version__, scipy=scipy.__version__, pandas=pd.__version__)
print(f"Huella de los datos: {DATA_HASH} | {len(NAMES)} galaxias | versiones: {VERS}")

def group_of(name):
    return "descubrimiento" if int(hashlib.md5((SPLIT_SALT + name).encode()).hexdigest(), 16) % 2 == 0 else "reservada"
GROUP = np.array([group_of(k) for k in NAMES])
DISC, HOLD = np.where(GROUP == "descubrimiento")[0], np.where(GROUP == "reservada")[0]
print(f"Descubrimiento: {len(DISC)} galaxias | Reservada: {len(HOLD)} galaxias")

## 2. Dos implementaciones geométricas de la no localidad

1. **Cascarones esféricos** (la de los notebooks 4–6): la masa fantasma de la RAR se reparte en 3D con el núcleo $e^{-d/\lambda}/(4\pi\lambda^2 d)$.
2. **Suavizado en el plano del disco** (nueva): se suaviza directamente la aceleración extra a lo largo del radio, $g_{ph,\lambda}(r) = \int g_{ph}(r')\,w_\lambda(r,r')\,dr'$, con $w \propto e^{-|r-r'|/\lambda} + e^{-(r+r')/\lambda}$ (reflejada en el centro). Fuera del último radio medido, la masa visible se extiende con caída kepleriana.

Las dos coinciden con la RAR local cuando λ → 0. Si la escala es real, las dos deberían dar una λ parecida. Si solo una la encuentra, la señal probablemente sea un artefacto de la geometría elegida.

In [ ]:
LAM_GRID = np.r_[0.0, np.geomspace(0.02, 200, 45)]
LOGL = np.log(np.r_[0.01, LAM_GRID[1:]])
BEAM_ARCSEC = 15.0
def chi_table(floor=0.0, cut_beams=None):
    TOT = np.zeros((len(NAMES), len(LAM_GRID), len(GD_GRID))); DAT = np.zeros_like(TOT); NPT = np.zeros(len(NAMES), int)
    for n_i, name in enumerate(NAMES):
        g = GAL[name]; s = setup(g)
        err = np.sqrt(g["err"] ** 2 + (floor * g["vobs"]) ** 2)
        mask = np.ones(len(g["r"]), bool)
        if cut_beams:
            mask = g["r"] >= cut_beams * BEAM_ARCSEC / ARCSEC * D[n_i] * 1e3
            if mask.sum() < 5:
                mask = np.zeros(len(g["r"]), bool); mask[-5:] = True
        NPT[n_i] = mask.sum()
        d = phantom_shells(s, U_GRID, GD_GRID)
        base = vb2(g, U_GRID[:, None]); pr = prior(U_GRID) ** 2
        for l_i, lam in enumerate(LAM_GRID):
            extra = np.einsum("in,ugn->ugi", F_matrix(g["r"], s["mids"], lam), d) / g["r"]
            v = np.sqrt(np.clip(base[:, None, :] + extra, 0, None))
            c = ((((g["vobs"] - v) / err) ** 2) * mask).sum(-1)
            t = c + pr[:, None]; k = t.argmin(0); a = np.arange(len(GD_GRID))
            TOT[n_i, l_i] = t[k, a]; DAT[n_i, l_i] = c[k, a]
    return dict(tot=TOT, dat=DAT, npts=NPT)



LAM_P = np.r_[0.0, np.geomspace(0.05, 20, 22)]
LOGLP = np.log(np.r_[0.01, LAM_P[1:]])
U_P = np.geomspace(0.1, 1.5, 24)
GD_P = GD_GRID[::2]
NGP = 300

def chi_table_plane(floor=0.05, cut_beams=None, gal=None):
    gal = GAL if gal is None else gal
    TOT = np.zeros((len(NAMES), len(LAM_P), len(GD_P))); DAT = np.zeros_like(TOT); NPT = np.zeros(len(NAMES), int)
    pr = prior(U_P) ** 2
    for n_i, name in enumerate(NAMES):
        g = gal[name]; r = g["r"]; rmax = r.max()
        err = np.sqrt(g["err"] ** 2 + (floor * g["vobs"]) ** 2)
        mask = np.ones(len(r), bool)
        if cut_beams:
            mask = r >= cut_beams * BEAM_ARCSEC / ARCSEC * D[n_i] * 1e3
            if mask.sum() < 5: mask = np.zeros(len(r), bool); mask[-5:] = True
        NPT[n_i] = mask.sum()
        base = vb2(g, U_P[:, None])                                                  # u, N
        gb_d = np.clip(base, 1e-6, None) / r
        for l_i, lam in enumerate(LAM_P):
            if lam == 0:
                gph = rar_g(gb_d[:, None, :], GD_P[None, :, None]) - gb_d[:, None, :]  # u, gd, N
                extra = gph * r
            else:
                rg = np.linspace(0, rmax + 5 * lam, NGP)[1:]
                def ext(comp):
                    inside = np.interp(rg, np.r_[0.0, r], np.r_[0.0, g[comp]])
                    return np.where(rg <= rmax, inside, g[comp][-1] * rmax / rg)
                v2g = ext("vgas2")[None, :] + U_P[:, None] * ext("vstar2")[None, :]
                gbg = np.clip(v2g, 1e-6, None) / rg
                gph = rar_g(gbg[:, None, :], GD_P[None, :, None]) - gbg[:, None, :]    # u, gd, NG
                W = np.exp(-np.abs(r[:, None] - rg[None, :]) / lam) + np.exp(-(r[:, None] + rg[None, :]) / lam)
                W /= W.sum(1, keepdims=True)
                extra = np.einsum("in,ugn->ugi", W, gph) * r
            v = np.sqrt(np.clip(base[:, None, :] + extra, 0, None))
            c = ((((g["vobs"] - v) / err) ** 2) * mask).sum(-1)
            t = c + pr[:, None]; k = t.argmin(0); a = np.arange(len(GD_P))
            TOT[n_i, l_i] = t[k, a]; DAT[n_i, l_i] = c[k, a]
    return dict(tot=TOT, dat=DAT, npts=NPT)

t0 = time.time()
TABS = {"esférica, 5 %": chi_table(floor=0.05), "esférica, 5 % sin interior": chi_table(floor=0.05, cut_beams=2)}
print(f"Tablas esféricas: {time.time()-t0:.0f} s"); t0 = time.time()
TABS["plano, 5 %"] = chi_table_plane(0.05); TABS["plano, 5 % sin interior"] = chi_table_plane(0.05, 2)
print(f"Tablas en el plano: {time.time()-t0:.0f} s")
GRIDS = {k: ((LAM_GRID, LOGL, GD_GRID) if k.startswith("esf") else (LAM_P, LOGLP, GD_P)) for k in TABS}

## 3. Ajuste en el grupo de descubrimiento

Para cada tabla se ajustan con las galaxias de descubrimiento:
- **RAR local** (λ = 0): solo g†;
- **λ constante**: λ y g†;
- **MOND completo**: λ = 0.27·R_d fijo, con g† ajustado;
- **Combinado**: λ = √((0.27·R_d)² + λ_f²), con λ_f y g† ajustados.

La incertidumbre de λ sale de un bootstrap de galaxias (1000 réplicas).

In [ ]:
def per_gal(tab, key, lam_vec, which="tot"):
    lams, logl, _ = GRIDS[key]
    T = tab[which]; n = np.arange(T.shape[0])
    ll = np.log(np.clip(lam_vec, 0.01, lams.max()))
    k = np.clip(np.searchsorted(logl, ll), 1, len(logl) - 1)
    w = np.clip((ll - logl[k - 1]) / (logl[k] - logl[k - 1]), 0, 1)[:, None]
    return T[n, k - 1, :] * (1 - w) + T[n, k, :] * w                      # galaxias × g†

def fit_model(tab, key, idx, kind):
    lams = GRIDS[key][0]
    if kind == "local":
        S = per_gal(tab, key, np.zeros(len(NAMES)))[idx].sum(0); j = int(np.argmin(S)); return dict(lam=0.0, lf=None, j=j, tot=S[j])
    if kind == "mond":
        S = per_gal(tab, key, ALPHA_Q * RD)[idx].sum(0); j = int(np.argmin(S)); return dict(lam=None, lf=None, j=j, tot=S[j])
    best = None
    cand = lams[1:] if kind == "const" else np.r_[0, np.geomspace(0.02, 20, 40)]
    for L in cand:
        lv = np.full(len(NAMES), L) if kind == "const" else np.sqrt((ALPHA_Q * RD) ** 2 + L ** 2)
        S = per_gal(tab, key, lv)[idx].sum(0); j = int(np.argmin(S))
        if best is None or S[j] < best["tot"]:
            best = dict(lam=L if kind == "const" else None, lf=L if kind == "comb" else None, j=j, tot=S[j])
    return best

def lam_vec(kind, p):
    if kind == "local": return np.zeros(len(NAMES))
    if kind == "const": return np.full(len(NAMES), p["lam"])
    if kind == "mond": return ALPHA_Q * RD
    return np.sqrt((ALPHA_Q * RD) ** 2 + p["lf"] ** 2)

FITS, BOOT = {}, {}
for key, tab in TABS.items():
    FITS[key] = {kd: fit_model(tab, key, DISC, kd) for kd in ["local", "const", "mond", "comb"]}
    bl = []
    for _ in range(1000):
        sub = RNG.choice(DISC, len(DISC))
        S = tab["tot"][sub].sum(0); bl.append(GRIDS[key][0][np.unravel_index(np.argmin(S), S.shape)[0]])
    BOOT[key] = np.array(bl)
    f = FITS[key]; gd = GRIDS[key][2]
    print(f"[{key}] λ = {f['const']['lam']:.3g} kpc (bootstrap 68 %: {np.percentile(bl,16):.2f}–{np.percentile(bl,84):.2f}; "
          f"λ = 0 en {np.mean(np.array(bl)==0):.0%}) | g† = {gd[f['const']['j']]*TO_MS2:.3g} | λ_f combinado = {f['comb']['lf']:.3g} kpc")

### 3.1 Calibración cruzada entre geometrías

Las dos geometrías no miden λ en la misma escala: suavizar la aceleración en el plano no es lo mismo que repartir masa en 3D. En pruebas con datos simulados, una λ esférica de 0.7 kpc aparece en el plano como ~3–4 kpc. Para compararlas, se **inyecta** la λ esférica medida en las mismas galaxias (con su masa visible real y sus errores), se ajusta la geometría en el plano a esos datos simulados y se obtiene la λ *esperada* en el plano. Si la λ medida en el plano con los datos reales coincide con la esperada, las dos geometrías cuentan la misma historia.

In [ ]:
f = FITS["esférica, 5 %"]; lam_s, gd_s = f["const"]["lam"], GD_GRID[f["const"]["j"]]
rng_c = np.random.default_rng(SEED + 1)
GAL_INJ = {}
for k in NAMES:
    g = dict(GAL[k]); v = np.sqrt(np.clip(model_v2(GAL[k], 0.5, lam_s, gd_s), 0, None))
    g["vobs"] = v + rng_c.normal(0, GAL[k]["err"]); GAL_INJ[k] = g
t0 = time.time()
TAB_INJ = chi_table_plane(0.05, gal=GAL_INJ)
S = TAB_INJ["tot"][DISC].sum(0); LAM_P_EXPECTED = float(LAM_P[np.unravel_index(np.argmin(S), S.shape)[0]])
lp_real = FITS["plano, 5 %"]["const"]["lam"]
print(f"λ esférica medida: {lam_s:.2f} kpc → λ esperada en el plano: {LAM_P_EXPECTED:.2f} kpc | λ medida en el plano: {lp_real:.2f} kpc  ({time.time()-t0:.0f} s)")

## 4. La prueba decisiva: el grupo reservado, con parámetros congelados

Para cada galaxia reservada se calcula $\Delta\chi^2_i = \chi^2_i(\text{rival}) - \chi^2_i(\lambda\ \text{constante})$, con los parámetros globales aprendidos en descubrimiento y solo Υ libre en cada galaxia. Positivo = la λ constante predice mejor esa galaxia.

- **Wilcoxon de una cola** sobre los $\Delta\chi^2_i$: ¿mejora la λ constante de forma sistemática, galaxia a galaxia, y no por unas pocas?
- Como no se ajusta nada en este grupo, no hay penalización por parámetros: es una predicción pura.

In [ ]:
HELD = {}
rows = []
for key, tab in TABS.items():
    f = FITS[key]
    chi = {kd: per_gal(tab, key, lam_vec(kd, f[kd]), "dat")[:, f[kd]["j"]] for kd in f}
    HELD[key] = chi
    for rival in ["local", "mond", "comb"]:
        d = (chi[rival] - chi["const"])[HOLD]
        stat = wilcoxon(d, alternative="greater") if np.any(d != 0) else None
        rows.append(dict(condiciones=key, rival={"local": "RAR local (λ = 0)", "mond": "MOND completo (0.27·R_d)", "comb": "Combinado MOND + λ_f"}[rival],
                         Δχ2_total=round(d.sum(), 1), galaxias_mejor=f"{np.mean(d > 0):.0%}",
                         p_Wilcoxon=(f"{stat.pvalue:.1e}" if stat is not None else "–")))
HT = pd.DataFrame(rows)
print("Grupo reservado (positivo = gana la λ constante):")
print(HT.to_string(index=False))

## 5. Controles instrumentales y de muestra

Si la mejora que aporta λ depende de cómo se observó cada galaxia, es sospechosa. Para todas las galaxias (con la tabla esférica al 5 % y parámetros de descubrimiento) se correlaciona la mejora por punto, $\Delta\chi^2_i / N_i$, con:
- **inclinación** (las galaxias muy inclinadas sufren más proyección);
- **distancia** (resolución física del haz);
- **muestreo** (puntos por kpc);
- **calidad** de SPARC (Q = 1 o 2);
- **fracción de gas** en el último radio (más gas = más extendido y más difícil de modelar).

In [ ]:
key0 = "esférica, 5 %"; chi = HELD[key0]
dpp = (chi["local"] - chi["const"]) / np.maximum(TABS[key0]["npts"], 1)
samp = np.array([len(GAL[k]["r"]) / GAL[k]["r"].max() for k in NAMES])
fgas = np.array([GAL[k]["vgas2"][-1] / max(GAL[k]["vgas2"][-1] + 0.5 * GAL[k]["vstar2"][-1], 1e-6) for k in NAMES])
CTRL = []
for nm, v in [("inclinación", INC), ("distancia", D), ("muestreo (puntos/kpc)", samp), ("calidad Q", QF), ("fracción de gas", fgas)]:
    ok = np.isfinite(v)
    if ok.sum() > 10 and np.nanstd(v[ok]) > 0:
        rho, p = spearmanr(v[ok], dpp[ok]); CTRL.append(dict(indicador=nm, ρ_Spearman=round(rho, 2), p=f"{p:.1e}", galaxias=int(ok.sum())))
    else:
        CTRL.append(dict(indicador=nm, ρ_Spearman=np.nan, p="sin datos", galaxias=int(ok.sum())))
CT = pd.DataFrame(CTRL)
print(CT.to_string(index=False))
print(f"\nMejora media por punto: {dpp.mean():.3f} | galaxias que mejoran: {np.mean(dpp > 0):.0%}")

## 6. El límite del modelo con κ constante (señalado en la revisión)

Si la masa de la red es κ veces la masa visible, repartida con un núcleo normalizado de alcance finito, la masa total es finita: $M_{tot} = (1+\kappa)M_b$. Más allá de unos pocos λ del borde, la velocidad vuelve a caer como $v \propto r^{-1/2}$ y **no puede sostener una curva plana indefinidamente**. Por eso la red con κ constante falló, y por eso el modelo final necesitó la forma de la RAR. La pregunta física abierta es generar ese comportamiento sin importarlo de una relación empírica.

In [ ]:
r_ = np.linspace(0.2, 200, 800); Rd_, Md_ = 3.0, 5e10
v2b = _v2_expdisk(r_, Md_, Rd_)
s_ = dict(rg=np.linspace(200 / 800, 200, 800))
GMb = np.interp(s_["rg"], r_, r_ * v2b); dGM = np.diff(np.r_[0, GMb]); mids = 0.5 * (np.r_[0, s_["rg"][:-1]] + s_["rg"])
fig, ax = plt.subplots(figsize=(8, 4))
for kap, lam, c in [(5.6, 2.3, "#eda100"), (5.6, 10.0, "#eb6834")]:
    v = np.sqrt(v2b + kap * (F_matrix(r_, mids, lam) @ dGM) / r_)
    ax.loglog(r_, v, color=c, label=f"red ER, κ = {kap}, λ = {lam} kpc")
vk = np.sqrt(G * Md_ * 6.6 / r_); ax.loglog(r_, vk, "--", color="#9a9a95", lw=1, label="∝ r^(−1/2) con masa (1+κ)·M_b")
vrar = np.sqrt(r_ * rar_g(np.clip(v2b, 1e-6, None) / r_, 1.0e-10 / TO_MS2))
ax.loglog(r_, vrar, color="#1baf7a", label="RAR (curva plana)")
ax.set_xlabel("r [kpc]"); ax.set_ylabel("v [km/s]"); ax.set_ylim(30, 600)
ax.set_title("Con κ constante la curva acaba cayendo; la RAR se mantiene plana", loc="left"); ax.legend(fontsize=8.5)
plt.show()

## 7. Muestra externa (cuando la tengas)

Una validación de verdad necesita galaxias que no hayan intervenido en nada de lo anterior, mejor con curvas de rotación de mayor resolución. Sube un `.zip` con archivos en el mismo formato que SPARC (`*_rotmod.dat`: radio, V_obs, error, V_gas, V_disk, V_bul). Se les aplican los parámetros **congelados** del grupo de descubrimiento (tabla esférica al 5 %) y se compara la λ constante con la RAR local. Si no hay muestra, la celda lo dice y continúa.

In [ ]:
EXT_DIR = "muestra_externa"
def load_external():
    os.makedirs(EXT_DIR, exist_ok=True)
    fs = glob.glob(os.path.join(EXT_DIR, "**", "*_rotmod.dat"), recursive=True)
    if not fs:
        try:
            from google.colab import files
            print("Opcional: sube un .zip con la muestra externa (o pulsa Cancelar para saltar este paso)")
            for nm, content in files.upload().items():
                zipfile.ZipFile(io.BytesIO(content)).extractall(EXT_DIR)
            fs = glob.glob(os.path.join(EXT_DIR, "**", "*_rotmod.dat"), recursive=True)
        except Exception:
            pass
    return {load_rotmod(f)["name"]: load_rotmod(f) for f in fs}

def fit_u_frozen(g, lam, gd, floor=0.05):
    err = np.sqrt(g["err"] ** 2 + (floor * g["vobs"]) ** 2)
    best = None
    for u in np.geomspace(0.1, 1.5, 60):
        v = np.sqrt(np.clip(model_v2(g, u, lam, gd), 0, None))
        c = float((((g["vobs"] - v) / err) ** 2).sum()); t = c + prior(u) ** 2
        if best is None or t < best[0]: best = (t, c)
    return best[1]

EXTERNAL = load_external()
EXT_RES = None
if EXTERNAL:
    f = FITS[key0]; lam_e, gd_e, gd_0 = f["const"]["lam"], GD_GRID[f["const"]["j"]], GD_GRID[f["local"]["j"]]
    rows = []
    for nm, g in EXTERNAL.items():
        if len(g["r"]) < 5: continue
        rows.append(dict(galaxia=nm, Δχ2=fit_u_frozen(g, 0.0, gd_0) - fit_u_frozen(g, lam_e, gd_e)))
    EXT_RES = pd.DataFrame(rows)
    st = wilcoxon(EXT_RES.Δχ2, alternative="greater")
    print(f"Muestra externa: {len(EXT_RES)} galaxias | Δχ² total = {EXT_RES.Δχ2.sum():.1f} | mejoran {np.mean(EXT_RES.Δχ2>0):.0%} | p Wilcoxon = {st.pvalue:.1e}")
else:
    print("Sin muestra externa: este paso queda pendiente.")

## 8. Veredicto y archivos de resultados

In [ ]:
def veredicto():
    if not HAVE_SPARC:
        print("⚠️  Datos SINTÉTICOS: solo comprueba que el código funciona.\n")
    def p_of(key, rival):
        v = HT[(HT.condiciones == key) & (HT.rival.str.startswith(rival))].p_Wilcoxon.iloc[0]
        return 1.0 if v == "–" else float(v)
    def d_of(key, rival):
        return float(HT[(HT.condiciones == key) & (HT.rival.str.startswith(rival))].Δχ2_total.iloc[0])
    c1 = (p_of("esférica, 5 %", "RAR local") < 0.01 and d_of("esférica, 5 %", "RAR local") > 0 and
          p_of("esférica, 5 % sin interior", "RAR local") < 0.05 and d_of("esférica, 5 % sin interior", "RAR local") > 0)
    c2 = all(d_of(k, r) > 0 and p_of(k, r) < 0.05 for k in ["esférica, 5 %", "esférica, 5 % sin interior"] for r in ["MOND", "Combinado"])
    ls, lp = FITS["esférica, 5 %"]["const"]["lam"], FITS["plano, 5 %"]["const"]["lam"]
    c3 = (lp > 0) and (LAM_P_EXPECTED > 0) and (max(LAM_P_EXPECTED, lp) / min(LAM_P_EXPECTED, lp) < 2.0)
    sig = CT[pd.to_numeric(CT.ρ_Spearman, errors="coerce").abs() > 0.3]
    sig = sig[[p != "sin datos" and float(p) < 0.01 for p in sig.p]]
    c4 = len(sig) == 0
    print(f"1) Predice mejor que la RAR local en el grupo reservado (p < 0.01; sin interior p < 0.05) → {'✓' if c1 else '✗'}")
    print(f"2) Predice mejor que MOND completo y que el modelo combinado → {'✓' if c2 else '✗'}")
    print(f"3) Geometría en el plano: λ medida {lp:.2f} kpc frente a esperada {LAM_P_EXPECTED:.2f} kpc → {'✓' if c3 else '✗'}")
    print(f"4) Sin correlación con indicadores instrumentales → {'✓' if c4 else '✗ (' + ', '.join(sig.indicador) + ')'}")
    ok = c1 and c2 and c3 and c4
    print()
    if ok:
        print(f"→ ESCALA CANDIDATA CONFIRMADA en validación interna: λ ≈ {ls:.2f} kpc predice galaxias que no se usaron para medirla, "
              "mejor que la RAR local y que MOND completo, con dos geometrías y sin rastro instrumental. Falta la muestra externa.")
    elif c1:
        print("→ SEÑAL PREDICTIVA PARCIAL: λ mejora la predicción fuera de muestra, pero falla alguna de las otras condiciones (ver arriba).")
    else:
        print("→ NO SE CONFIRMA: con parámetros congelados, λ no predice mejor las galaxias reservadas. La señal de los notebooks anteriores "
              "probablemente era específica de la muestra o de la forma de ajustar.")
    return dict(c1=c1, c2=c2, c3=c3, c4=c4, confirmada=ok)

V = veredicto()
out = dict(huella_datos=DATA_HASH, versiones=VERS, semilla=SEED, sal_division=SPLIT_SALT, alpha_mond=ALPHA_Q,
           galaxias=dict(descubrimiento=len(DISC), reservada=len(HOLD)),
           parametros={k: dict(lam=FITS[k]["const"]["lam"], g_dagger=float(GRIDS[k][2][FITS[k]["const"]["j"]] * TO_MS2),
                               lam_f_combinado=FITS[k]["comb"]["lf"], lam_plano_esperada=LAM_P_EXPECTED,
                               bootstrap_68=[float(np.percentile(BOOT[k], 16)), float(np.percentile(BOOT[k], 84))]) for k in TABS},
           reservada=HT.to_dict(orient="records"), controles=CT.to_dict(orient="records"), veredicto=V,
           muestra_externa=None if EXT_RES is None else EXT_RES.to_dict(orient="records"))
json.dump(out, open("resultados_validacion.json", "w"), ensure_ascii=False, indent=1, default=float)
pd.DataFrame(dict(galaxia=NAMES, grupo=GROUP, D_Mpc=D, R_d=RD, inclinacion=INC, Q=QF,
                  dchi2_local_menos_const=chi["local"] - chi["const"], dchi2_mond_menos_const=chi["mond"] - chi["const"],
                  puntos=TABS[key0]["npts"])).to_csv("por_galaxia.csv", index=False)
print("\nGuardado: resultados_validacion.json y por_galaxia.csv")
try:
    from google.colab import files
    files.download("resultados_validacion.json"); files.download("por_galaxia.csv")
except Exception:
    pass